# Healthier / NutriLabel — Revision Training Notebook (Kaggle T4)

Notebook ini adalah pipeline **revisi** yang reproducible untuk:

1. Membangun dataset segmentasi 5 kelas Isi Piringku dari UECFoodPIXCOMPLETE,
   FruitSeg30, dan FoodSeg103.
2. Membuat split train/val + **tiga external test** yang tidak dipakai training:
   UEC test, FoodSeg validation, dan FruitSeg30.
3. Screening arsitektur (opsional), training utama YOLO11s-seg, dan baseline.
4. Evaluasi **instance-level** (mAP) dan **pixel-level** (TP/FP/FN/TN,
   confusion matrix) untuk setiap external test.
5. Evaluasi opsional 10–20 foto piring Indonesia (LabelMe/YOLO) sebagai
   *exploratory domain-shift analysis*.

## Sebelum Run

- Tambahkan Kaggle Dataset:
  - `UECFoodPIXCOMPLETE`
  - `FruitSeg30`
- Aktifkan **Internet** untuk mengunduh FoodSeg103 dari HuggingFace.
- Pilih **GPU T4** dan **Persistence: Files**.
- Notebook ini **tidak** memakai 19.087 sebagai “training”; jumlah train/val/test
  akan dicetak di cell dataset summary.

## Flag penting (cell CONFIG)

| Flag | Arti |
|---|---|
| `RUN_SCREENING` | 3 model × 5 epoch untuk shortlist (appendix, bukan final). |
| `RUN_MAIN` | Training utama YOLO11s-seg 100 epoch/patience 20. |
| `RUN_BASELINE` | Baseline full-budget YOLO11n-seg. Jalankan di sesi terpisah jika waktu habis. |
| `RUN_EXTERNAL_EVAL` | Evaluasi UEC test, FoodSeg val, FruitSeg30. |
| `RUN_INDONESIAN_EVAL` | Evaluasi 10–20 foto Indonesia jika tersedia. |
| `FORCE_TRAIN` | Paksa training ulang walau `best.pt` sudah ada. |

> **Catatan waktu:** dataset prep ±20–40 menit; main training ±6–9 jam di T4;
> baseline ±3–5 jam. Jika sesi Kaggle hampir habis, set `RUN_MAIN=False` dan
> jalankan baseline di sesi berikutnya.

## 1. Install, import, dan verifikasi GPU

In [ ]:
import os, sys, json, random, shutil, subprocess, warnings, gc, csv, zipfile
from pathlib import Path
from collections import Counter, defaultdict

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4.0", "datasets>=4.0.0"],
    check=False,
)

import numpy as np
import cv2
from PIL import Image
import torch
import yaml

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM GB:", round(props.total_memory / 1e9, 1))
else:
    print("GPU tidak terdeteksi. Aktifkan GPU T4 di Kaggle.")

print("PyTorch:", torch.__version__)

## 2. CONFIG — ubah di sini saja

In [ ]:
# ─── Mode ────────────────────────────────────────────────────────────────────
RUN_SCREENING = False
RUN_MAIN = True
RUN_BASELINE = False
RUN_EXTERNAL_EVAL = True
RUN_INDONESIAN_EVAL = True
FORCE_TRAIN = False

# ─── Path ────────────────────────────────────────────────────────────────────
WORK = Path("/kaggle/working")
DATA_ROOT = WORK / "revision_dataset"
EXTERNAL_ROOT = WORK / "external"
RUNS = WORK / "runs"
RESULTS = WORK / "results"
for folder in (DATA_ROOT, EXTERNAL_ROOT, RUNS, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

CLEAR_WORK = False  # True jika ingin menghapus dataset hasil run sebelumnya
if CLEAR_WORK:
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    shutil.rmtree(EXTERNAL_ROOT, ignore_errors=True)
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)

# Isi manual jika auto-detect gagal:
UEC_ROOT_CFG = None
FRUITSEG_ROOT_CFG = None
INDONESIAN_ROOT = Path("/kaggle/input/indonesian-plates")  # opsional

# ─── Dataset / training ─────────────────────────────────────────────────────
CLASS_NAMES = ["Staple Food", "Side Dishes", "Vegetables", "Fruits", "Others"]
NUM_CLASSES = len(CLASS_NAMES)
VAL_FRAC = 0.15
MIN_AREA_PX = 100
EPSILON = 0.005
IMG_SIZE = 640
BATCH = 16
WORKERS = 2
EPOCHS = 100
PATIENCE = 20

MAIN_YAML = DATA_ROOT / "data.yaml"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Working dir:", WORK)
print("Kelas:", CLASS_NAMES)
print("Device:", DEVICE)

## 3. Mapping kelas (diambil dari notebook asli)

- UECFood: 102 kelas → 5 kelompok.
- FoodSeg103: 104 kelas → 5 kelompok.
- FruitSeg30: semua folder buah → kelas 3 (`Fruits`).

In [ ]:
UECFOOD_TO_ISIPIRINGKU = {1: 0,
 2: 0,
 3: 0,
 4: 0,
 5: 0,
 6: 4,
 7: 0,
 8: 0,
 9: 0,
 10: 0,
 11: 0,
 12: 0,
 13: 0,
 14: 0,
 15: 0,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: 0,
 21: 0,
 22: 0,
 23: 0,
 24: 0,
 25: 0,
 26: 0,
 27: 0,
 28: 0,
 29: 0,
 30: 4,
 31: 2,
 32: 2,
 33: 2,
 34: 2,
 35: 2,
 36: 4,
 37: 4,
 38: 1,
 39: 2,
 40: 1,
 41: 1,
 42: 1,
 43: 4,
 44: 1,
 45: 1,
 46: 1,
 47: 1,
 48: 1,
 49: 1,
 50: 1,
 51: 1,
 52: 1,
 53: 1,
 54: 1,
 55: 1,
 56: 1,
 57: 1,
 58: 1,
 59: 1,
 60: 1,
 61: 1,
 62: 1,
 63: 1,
 64: 1,
 65: 1,
 66: 1,
 67: 1,
 68: 1,
 69: 1,
 70: 1,
 71: 1,
 72: 0,
 73: 1,
 74: 1,
 75: 1,
 76: 1,
 77: 1,
 78: 4,
 79: 1,
 80: 1,
 81: 1,
 82: 0,
 83: 0,
 84: 0,
 85: 1,
 86: 2,
 87: 2,
 88: 2,
 89: 4,
 90: 4,
 91: 4,
 92: 0,
 93: 2,
 94: 0,
 95: 0,
 96: 0,
 97: 0,
 98: 1,
 99: 0,
 100: 2,
 101: 4,
 102: 4}

FOODSEG_TO_ISIPIRINGKU = {0: 4,
 1: 4,
 2: 4,
 3: 0,
 4: 4,
 5: 4,
 6: 4,
 7: 4,
 8: 4,
 9: 4,
 10: 4,
 11: 4,
 12: 0,
 13: 4,
 14: 4,
 15: 4,
 16: 4,
 17: 4,
 18: 4,
 19: 4,
 20: 4,
 21: 4,
 22: 4,
 23: 4,
 24: 1,
 25: 3,
 26: 3,
 27: 3,
 28: 3,
 29: 3,
 30: 3,
 31: 3,
 32: 3,
 33: 3,
 34: 3,
 35: 3,
 36: 3,
 37: 3,
 38: 3,
 39: 3,
 40: 3,
 41: 3,
 42: 3,
 43: 3,
 44: 3,
 45: 3,
 46: 1,
 47: 1,
 48: 1,
 49: 1,
 50: 1,
 51: 1,
 52: 4,
 53: 1,
 54: 1,
 55: 1,
 56: 1,
 57: 4,
 58: 0,
 59: 0,
 60: 0,
 61: 0,
 62: 0,
 63: 0,
 64: 0,
 65: 0,
 66: 0,
 67: 0,
 68: 1,
 69: 2,
 70: 2,
 71: 2,
 72: 2,
 73: 2,
 74: 2,
 75: 2,
 76: 2,
 77: 2,
 78: 2,
 79: 2,
 80: 2,
 81: 2,
 82: 2,
 83: 2,
 84: 2,
 85: 2,
 86: 2,
 87: 2,
 88: 2,
 89: 2,
 90: 2,
 91: 2,
 92: 2,
 93: 2,
 94: 2,
 95: 2,
 96: 2,
 97: 2,
 98: 2,
 99: 2,
 100: 2,
 101: 2,
 102: 2,
 103: 4}

FOODSEG103_NAMES = {0: 'background',
 1: 'candy',
 2: 'chocolate',
 3: 'french fries',
 4: 'cream',
 5: 'biscuit',
 6: 'jam',
 7: 'honey',
 8: 'sugar',
 9: 'vinegar',
 10: 'oil',
 11: 'condiment',
 12: 'cooked rice',
 13: 'soy sauce',
 14: 'sake',
 15: 'mirin',
 16: 'sweet bean paste',
 17: 'dashi',
 18: 'pickled ginger',
 19: 'wasabi',
 20: 'mayonnaise',
 21: 'ketchup',
 22: 'salad dressing',
 23: 'peanut butter',
 24: 'egg',
 25: 'apple',
 26: 'date',
 27: 'apricot',
 28: 'avocado',
 29: 'banana',
 30: 'strawberry',
 31: 'cherry',
 32: 'blueberry',
 33: 'raspberry',
 34: 'mango',
 35: 'olives',
 36: 'peach',
 37: 'lemon',
 38: 'pear',
 39: 'fig',
 40: 'pineapple',
 41: 'grape',
 42: 'kiwi',
 43: 'melon',
 44: 'orange',
 45: 'watermelon',
 46: 'steak',
 47: 'pork',
 48: 'chicken duck',
 49: 'sausage',
 50: 'fried meat',
 51: 'lamb',
 52: 'other seafood',
 53: 'crab',
 54: 'fish',
 55: 'shellfish',
 56: 'shrimp',
 57: 'sauce',
 58: 'bread',
 59: 'corn',
 60: 'hamburg',
 61: 'pizza',
 62: 'hanamaki baozi',
 63: 'wonton dumplings',
 64: 'pasta',
 65: 'noodles',
 66: 'rice',
 67: 'pie',
 68: 'tofu',
 69: 'eggplant',
 70: 'potato',
 71: 'garlic',
 72: 'cauliflower',
 73: 'tomato',
 74: 'kelp',
 75: 'seaweed',
 76: 'spring onion',
 77: 'rape',
 78: 'ginger',
 79: 'okra',
 80: 'lettuce',
 81: 'pumpkin',
 82: 'cucumber',
 83: 'white radish',
 84: 'carrot',
 85: 'asparagus',
 86: 'bamboo shoots',
 87: 'broccoli',
 88: 'celery',
 89: 'cilantro mint',
 90: 'snow peas',
 91: 'cabbage',
 92: 'bean sprouts',
 93: 'onion',
 94: 'pepper',
 95: 'green beans',
 96: 'french beans',
 97: 'king oyster mushroom',
 98: 'shiitake',
 99: 'enoki mushroom',
 100: 'oyster mushroom',
 101: 'white button mushroom',
 102: 'salad',
 103: 'coconut'}

print('UEC mapping:', len(UECFOOD_TO_ISIPIRINGKU))
print('FoodSeg mapping:', len(FOODSEG_TO_ISIPIRINGKU))

## 4. Helper: auto-detect dataset, mask → polygon, split, manifest

In [ ]:
def find_uec_root(base="/kaggle/input"):
    candidates = [p for p in Path(base).rglob("UECFoodPIXCOMPLETE") if p.is_dir()]
    for p in candidates:
        if (p / "train" / "img").is_dir() and (p / "test" / "img").is_dir():
            return p
    return candidates[0] if candidates else None


def find_fruitseg_root(base="/kaggle/input"):
    candidates = [p for p in Path(base).rglob("FruitSeg30") if p.is_dir()]
    for p in candidates:
        for child in p.iterdir():
            if child.is_dir() and (child / "Images").is_dir() and (child / "Mask").is_dir():
                return p
    return None


UEC_ROOT = Path(UEC_ROOT_CFG) if UEC_ROOT_CFG else find_uec_root()
FRUITSEG_ROOT = Path(FRUITSEG_ROOT_CFG) if FRUITSEG_ROOT_CFG else find_fruitseg_root()

print("UEC root      :", UEC_ROOT)
print("FruitSeg root :", FRUITSEG_ROOT)
assert UEC_ROOT and UEC_ROOT.is_dir(), "UECFoodPIXCOMPLETE tidak ditemukan. Isi UEC_ROOT_CFG."
assert FRUITSEG_ROOT and FRUITSEG_ROOT.is_dir(), "FruitSeg30 tidak ditemukan. Isi FRUITSEG_ROOT_CFG."


def ensure_2d_mask(mask_np):
    arr = np.asarray(mask_np)
    if arr.ndim == 3:
        # UEC-FoodPix: class id ada di channel R.
        arr = arr[:, :, 0]
    return arr.astype(np.int32)


def mask_to_polygons(mask_np, class_id, img_h, img_w, min_area=MIN_AREA_PX, epsilon=EPSILON):
    binary = (mask_np > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        approx = cv2.approxPolyDP(cnt, epsilon * cv2.arcLength(cnt, True), True)
        if len(approx) < 3:
            continue
        pts = approx.reshape(-1, 2).astype(float)
        pts[:, 0] /= img_w
        pts[:, 1] /= img_h
        pts = np.clip(pts, 0.0, 1.0)
        coords = " ".join(f"{x:.6f} {y:.6f}" for x, y in pts)
        lines.append(f"{class_id} {coords}")
    return lines


def dhash(img_pil, size=8):
    gray = np.asarray(img_pil.convert("L").resize((size + 1, size)))
    diff = gray[:, 1:] > gray[:, :-1]
    value = 0
    for bit in diff.flatten():
        value = (value << 1) | int(bit)
    return value


def write_data_yaml(path, root, train, val, test):
    data = {
        "path": str(root),
        "train": train,
        "val": val,
        "test": test,
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }
    path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return path


def count_labels(label_dir):
    instances = Counter()
    images_with = Counter()
    for label_path in sorted(Path(label_dir).glob("*.txt")):
        content = label_path.read_text(encoding="utf-8").strip()
        if not content:
            continue
        classes_in_file = set()
        for line in content.splitlines():
            parts = line.split()
            if len(parts) >= 7:
                try:
                    cls = int(parts[0])
                except ValueError:
                    continue
                if 0 <= cls < NUM_CLASSES:
                    instances[cls] += 1
                    classes_in_file.add(cls)
        for cls in classes_in_file:
            images_with[cls] += 1
    return instances, images_with


manifest = []
seen_train_hashes = {}
duplicates_skipped = 0

## 5. Proses UECFood: train/val + external test

In [ ]:
def find_uec_list(filename):
    for candidate in (UEC_ROOT / filename, UEC_ROOT.parent / filename, UEC_ROOT.parent.parent / filename):
        if candidate.is_file():
            return [line.strip() for line in candidate.read_text(encoding="utf-8").splitlines() if line.strip()]
    return None


uec_train_files = find_uec_list("train9000.txt")
uec_test_files = find_uec_list("test1000.txt")

if uec_train_files is None:
    uec_train_files = [p.stem for p in sorted((UEC_ROOT / "train" / "img").glob("*.jpg"))]
if uec_test_files is None:
    uec_test_files = [p.stem for p in sorted((UEC_ROOT / "test" / "img").glob("*.jpg"))]

rng = random.Random(SEED)
rng.shuffle(uec_train_files)
n_val = int(len(uec_train_files) * VAL_FRAC)
uec_val_files = uec_train_files[:n_val]
uec_train_files = uec_train_files[n_val:]

print("UEC train/val/test:", len(uec_train_files), len(uec_val_files), len(uec_test_files))


def process_uec(files, source_split, out_split, out_root, is_test=False):
    global duplicates_skipped
    img_out = out_root / out_split / "images"
    lbl_out = out_root / out_split / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    processed = 0
    for name in files:
        stem = Path(name).stem
        img_path = UEC_ROOT / source_split / "img" / f"{stem}.jpg"
        mask_path = UEC_ROOT / source_split / "mask" / f"{stem}.png"
        if not img_path.is_file() or not mask_path.is_file():
            continue

        img = Image.open(img_path).convert("RGB")
        h, w = img.size[1], img.size[0]
        mask = ensure_2d_mask(Image.open(mask_path))
        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        image_hash = dhash(img)
        if image_hash in seen_train_hashes:
            duplicates_skipped += 1
            continue
        if not is_test:
            seen_train_hashes[image_hash] = f"uec/{stem}"

        lines = []
        for orig_id in np.unique(mask):
            if int(orig_id) == 0:
                continue
            new_id = UECFOOD_TO_ISIPIRINGKU.get(int(orig_id), 4)
            single = np.where(mask == orig_id, 255, 0).astype(np.uint8)
            lines.extend(mask_to_polygons(single, new_id, h, w))

        out_stem = f"uec_{stem}"
        img.save(img_out / f"{out_stem}.jpg", quality=95)
        (lbl_out / f"{out_stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        manifest.append({"source": "uec", "split": out_split, "image": f"{out_split}/images/{out_stem}.jpg", "labels": len(lines)})
        processed += 1

    print(f"UEC {out_split}: {processed} gambar")
    return processed


process_uec(uec_train_files, "train", "train", DATA_ROOT)
process_uec(uec_val_files, "train", "val", DATA_ROOT)
process_uec(uec_test_files, "test", "test", EXTERNAL_ROOT / "uec_test", is_test=True)

write_data_yaml(EXTERNAL_ROOT / "uec_test" / "data.yaml", EXTERNAL_ROOT / "uec_test", "test/images", "test/images", "test/images")

## 6. Proses FoodSeg103 (HuggingFace): train/val + external test

In [ ]:
from datasets import load_dataset

print("Mengunduh FoodSeg103 dari HuggingFace...")
ds = None
for attempt in range(1, 4):
    try:
        ds = load_dataset("EduardoPacheco/FoodSeg103")
        break
    except Exception as exc:
        print(f"Percobaan {attempt}/3 gagal: {exc}")
        if attempt == 3:
            raise
print("FoodSeg train:", len(ds["train"]), "validation:", len(ds["validation"]))

food_train_indices = list(range(len(ds["train"])))
rng = random.Random(SEED)
rng.shuffle(food_train_indices)
n_val_food = int(len(food_train_indices) * VAL_FRAC)
food_val_indices = food_train_indices[:n_val_food]
food_train_indices = food_train_indices[n_val_food:]


def process_foodseg(indices, hf_split, out_split, out_root, is_test=False):
    global duplicates_skipped
    img_out = out_root / out_split / "images"
    lbl_out = out_root / out_split / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    processed = 0
    for idx in indices:
        sample = ds[hf_split][int(idx)]
        img = sample["image"].convert("RGB")
        w, h = img.size
        mask = ensure_2d_mask(np.array(sample["label"]))
        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        image_hash = dhash(img)
        if image_hash in seen_train_hashes:
            duplicates_skipped += 1
            continue
        if not is_test:
            seen_train_hashes[image_hash] = f"foodseg/{idx}"

        lines = []
        for orig_id in np.unique(mask):
            if int(orig_id) == 0:
                continue
            new_id = FOODSEG_TO_ISIPIRINGKU.get(int(orig_id), 4)
            single = np.where(mask == orig_id, 255, 0).astype(np.uint8)
            lines.extend(mask_to_polygons(single, new_id, h, w))

        out_stem = f"fseg_{out_split}_{int(idx):05d}"
        img.save(img_out / f"{out_stem}.jpg", quality=95)
        (lbl_out / f"{out_stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        manifest.append({"source": "foodseg", "split": out_split, "image": f"{out_split}/images/{out_stem}.jpg", "labels": len(lines)})
        processed += 1

    print(f"FoodSeg {out_split}: {processed} gambar")
    return processed


process_foodseg(food_train_indices, "train", "train", DATA_ROOT)
process_foodseg(food_val_indices, "train", "val", DATA_ROOT)
process_foodseg(list(range(len(ds["validation"]))), "validation", "test", EXTERNAL_ROOT / "foodseg_val", is_test=True)

write_data_yaml(EXTERNAL_ROOT / "foodseg_val" / "data.yaml", EXTERNAL_ROOT / "foodseg_val", "test/images", "test/images", "test/images")

del ds
gc.collect()

## 7. Proses FruitSeg30 sebagai external test (cross-dataset)

In [ ]:
def process_fruitseg(out_root):
    global duplicates_skipped
    img_out = out_root / "test" / "images"
    lbl_out = out_root / "test" / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    processed = 0
    for class_folder in sorted(FRUITSEG_ROOT.iterdir()):
        if not class_folder.is_dir():
            continue
        img_dir = class_folder / "Images"
        mask_dir = class_folder / "Mask"
        if not img_dir.is_dir() or not mask_dir.is_dir():
            continue

        for img_path in sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.JPG")) + list(img_dir.glob("*.png"))):
            mask_path = mask_dir / f"{img_path.stem}_mask.png"
            if not mask_path.is_file():
                mask_path = mask_dir / f"{img_path.stem}.png"
            if not mask_path.is_file():
                candidates = list(mask_dir.glob(f"{img_path.stem}*"))
                if not candidates:
                    continue
                mask_path = candidates[0]

            img = Image.open(img_path).convert("RGB")
            w, h = img.size
            mask = ensure_2d_mask(Image.open(mask_path))
            if mask.shape[:2] != (h, w):
                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            binary = np.where(mask > 127, 255, 0).astype(np.uint8)

            image_hash = dhash(img)
            if image_hash in seen_train_hashes:
                duplicates_skipped += 1
                continue

            lines = mask_to_polygons(binary, 3, h, w)  # 3 = Fruits
            safe_class = class_folder.name.replace(" ", "_").replace("&", "and")
            out_stem = f"fruit_{safe_class}_{img_path.stem}"
            img.save(img_out / f"{out_stem}.jpg", quality=95)
            (lbl_out / f"{out_stem}.txt").write_text("\n".join(lines), encoding="utf-8")
            manifest.append({"source": "fruitseg", "split": "test", "image": f"test/images/{out_stem}.jpg", "labels": len(lines)})
            processed += 1

    print("FruitSeg30 external test:", processed, "gambar")


process_fruitseg(EXTERNAL_ROOT / "fruitseg")
write_data_yaml(EXTERNAL_ROOT / "fruitseg" / "data.yaml", EXTERNAL_ROOT / "fruitseg", "test/images", "test/images", "test/images")

## 8. Ringkasan dataset, distribusi kelas, dan data.yaml

In [ ]:
write_data_yaml(MAIN_YAML, DATA_ROOT, "train/images", "val/images", "val/images")

train_instances, train_images = count_labels(DATA_ROOT / "train" / "labels")
val_instances, val_images = count_labels(DATA_ROOT / "val" / "labels")

external_summary = {}
for name in ("uec_test", "foodseg_val", "fruitseg"):
    root = EXTERNAL_ROOT / name
    instances, images_with = count_labels(root / "test" / "labels")
    external_summary[name] = {
        "instances": {CLASS_NAMES[i]: int(instances[i]) for i in range(NUM_CLASSES)},
        "images_with_class": {CLASS_NAMES[i]: int(images_with[i]) for i in range(NUM_CLASSES)},
        "total_images": len(list((root / "test" / "images").glob("*.jpg"))),
    }

summary = {
    "class_names": CLASS_NAMES,
    "main": {
        "train_images": len(list((DATA_ROOT / "train" / "images").glob("*.jpg"))),
        "val_images": len(list((DATA_ROOT / "val" / "images").glob("*.jpg"))),
        "train_instances": {CLASS_NAMES[i]: int(train_instances[i]) for i in range(NUM_CLASSES)},
        "train_images_with_class": {CLASS_NAMES[i]: int(train_images[i]) for i in range(NUM_CLASSES)},
        "val_instances": {CLASS_NAMES[i]: int(val_instances[i]) for i in range(NUM_CLASSES)},
        "val_images_with_class": {CLASS_NAMES[i]: int(val_images[i]) for i in range(NUM_CLASSES)},
    },
    "external": external_summary,
    "duplicates_skipped": duplicates_skipped,
    "mapping": {"uec": UECFOOD_TO_ISIPIRINGKU, "foodseg": FOODSEG_TO_ISIPIRINGKU},
}
(DATA_ROOT / "dataset_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
(DATA_ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(summary["main"], indent=2, ensure_ascii=False))
print("\nExternal tests:")
for name, info in external_summary.items():
    print(name, "images:", info["total_images"])
print("\nDuplicate images skipped:", duplicates_skipped)
print("Main data.yaml:", MAIN_YAML)

## 9. Screening arsitektur (opsional, appendix)

Semua model memakai budget yang sama (5 epoch) dan dievaluasi pada **validation**.
Ini bukan final comparison; hanya untuk shortlist.

In [ ]:
from ultralytics import YOLO

SCREEN_MODELS = ["yolo11s-seg.pt", "yolo11n-seg.pt", "yolov8s-seg.pt"]

if RUN_SCREENING:
    for model_name in SCREEN_MODELS:
        run_name = f"screen_{Path(model_name).stem}"
        print("\n" + "=" * 70)
        print("Screening:", model_name)
        print("=" * 70)
        model = YOLO(model_name)
        model.train(
            data=str(MAIN_YAML),
            epochs=5,
            imgsz=IMG_SIZE,
            batch=BATCH,
            device=DEVICE,
            workers=WORKERS,
            cache="disk",
            seed=SEED,
            project=str(RUNS),
            name=run_name,
            exist_ok=True,
            plots=True,
            verbose=True,
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("RUN_SCREENING=False — dilewati.")

In [ ]:
if RUN_SCREENING:
    screening_rows = []
    for run_dir in sorted(RUNS.glob("screen_*")):
        results_csv = run_dir / "results.csv"
        if not results_csv.is_file():
            continue
        rows = list(csv.DictReader(results_csv.open(encoding="utf-8")))
        if not rows:
            continue
        last = rows[-1]
        screening_rows.append({
            "run": run_dir.name,
            "epoch": int(last["epoch"]),
            "mask_map50": float(last["metrics/mAP50(M)"]),
            "mask_map50_95": float(last["metrics/mAP50-95(M)"]),
            "box_map50": float(last["metrics/mAP50(B)"]),
        })
    screening_rows.sort(key=lambda row: row["mask_map50_95"], reverse=True)
    for row in screening_rows:
        print(row)
    if screening_rows:
        print("\nTop-2 screening (pakai validation, bukan test):", screening_rows[:2])
else:
    print("RUN_SCREENING=False — tidak ada screening summary.")

## 10. Training utama — YOLO11s-seg (100 epoch, patience 20)

In [ ]:
main_best = RUNS / "seg_main" / "weights" / "best.pt"

if RUN_MAIN:
    if main_best.exists() and not FORCE_TRAIN:
        print("best.pt sudah ada, skip:", main_best)
    else:
        model = YOLO("yolo11s-seg.pt")
        model.train(
            data=str(MAIN_YAML),
            epochs=EPOCHS,
            patience=PATIENCE,
            imgsz=IMG_SIZE,
            batch=BATCH,
            device=DEVICE,
            workers=WORKERS,
            cache="disk",
            seed=SEED,
            project=str(RUNS),
            name="seg_main",
            exist_ok=True,
            optimizer="AdamW",
            lr0=0.001,
            lrf=0.01,
            momentum=0.9,
            weight_decay=0.0005,
            hsv_h=0.01,
            hsv_s=0.3,
            hsv_v=0.2,
            degrees=10.0,
            translate=0.1,
            scale=0.5,
            fliplr=0.5,
            mosaic=0.5,
            erasing=0.4,
            close_mosaic=10,
            plots=True,
            verbose=True,
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()

else:
    print("RUN_MAIN=False — dilewati.")

import ultralytics
main_results = RUNS / "seg_main" / "results.csv"
if main_results.is_file():
    rows = list(csv.DictReader(main_results.open(encoding="utf-8")))
    if rows:
        last = rows[-1]
        print("Main run last epoch:", last.get("epoch"))
        print("Mask mAP50:", last.get("metrics/mAP50(M)"))
        print("Mask mAP50-95:", last.get("metrics/mAP50-95(M)"))

training_config = {
    "seed": SEED,
    "class_names": CLASS_NAMES,
    "val_frac": VAL_FRAC,
    "min_area_px": MIN_AREA_PX,
    "epsilon": EPSILON,
    "imgsz": IMG_SIZE,
    "batch": BATCH,
    "workers": WORKERS,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "model": "yolo11s-seg.pt",
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "augmentation": {
        "hsv_h": 0.01, "hsv_s": 0.3, "hsv_v": 0.2,
        "degrees": 10.0, "translate": 0.1, "scale": 0.5,
        "fliplr": 0.5, "mosaic": 0.5, "erasing": 0.4,
        "close_mosaic": 10,
    },
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(DATA_ROOT / "training_config.json").write_text(
    json.dumps(training_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Training config saved:", DATA_ROOT / "training_config.json")

print("Main best:", main_best)
print("Main last:", RUNS / "seg_main" / "weights" / "last.pt")

## 11. Baseline full-budget (opsional)

Jika waktu sesi tidak cukup, set `RUN_MAIN=False`, `RUN_BASELINE=True`, lalu
jalankan di sesi Kaggle baru. Jangan memakai baseline 5 epoch sebagai tabel utama.

In [ ]:
baseline_best = RUNS / "baseline_11n" / "weights" / "best.pt"

if RUN_BASELINE:
    if baseline_best.exists() and not FORCE_TRAIN:
        print("Baseline sudah ada, skip:", baseline_best)
    else:
        model = YOLO("yolo11n-seg.pt")
        model.train(
            data=str(MAIN_YAML),
            epochs=EPOCHS,
            patience=PATIENCE,
            imgsz=IMG_SIZE,
            batch=BATCH,
            device=DEVICE,
            workers=WORKERS,
            cache="disk",
            seed=SEED,
            project=str(RUNS),
            name="baseline_11n",
            exist_ok=True,
            optimizer="AdamW",
            lr0=0.001,
            lrf=0.01,
            momentum=0.9,
            weight_decay=0.0005,
            hsv_h=0.01,
            hsv_s=0.3,
            hsv_v=0.2,
            degrees=10.0,
            translate=0.1,
            scale=0.5,
            fliplr=0.5,
            mosaic=0.5,
            erasing=0.4,
            close_mosaic=10,
            plots=True,
            verbose=True,
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("RUN_BASELINE=False — dilewati.")

## 12. Evaluasi: instance-level + pixel-level

- Instance-level: box/mask P/R/mAP50/mAP50-95.
- Pixel-level: TP/FP/FN/TN, accuracy, P/R/F1 per kelas, confusion matrix.

Keduanya dilaporkan terpisah; jangan dicampur.

In [ ]:
def load_names(data_yaml):
    data = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
    names = data.get("names")
    if isinstance(names, dict):
        return [str(names[i]) for i in sorted(names)]
    return [str(n) for n in names]


def yolo_txt_to_mask(label_path, height, width, num_classes):
    mask = np.zeros((height, width), dtype=np.uint8)
    if not Path(label_path).is_file():
        return mask
    content = Path(label_path).read_text(encoding="utf-8").strip()
    if not content:
        return mask
    for line in content.splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        cls_id = int(parts[0])
        if not (0 <= cls_id < num_classes):
            continue
        coords = list(map(float, parts[1:]))
        pts = [(int(coords[i] * width), int(coords[i + 1] * height)) for i in range(0, len(coords), 2)]
        cv2.fillPoly(mask, [np.array(pts, dtype=np.int32).reshape((-1, 1, 2))], cls_id + 1)
    return mask


def eval_instance(model_path, data_yaml, name, split="test"):
    from ultralytics import YOLO
    out_dir = RESULTS / name
    out_dir.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(model_path))
    metrics = model.val(
        data=str(data_yaml),
        split=split,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        conf=0.001,
        iou=0.6,
        plots=True,
        save_json=True,
        project=str(RESULTS),
        name=name,
        exist_ok=True,
        verbose=True,
    )
    names = load_names(data_yaml)
    summary = {
        "model": str(model_path),
        "data": str(data_yaml),
        "split": split,
        "box": {
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "map50": float(metrics.box.map50),
            "map50_95": float(metrics.box.map),
        },
        "mask": {
            "precision": float(metrics.seg.mp),
            "recall": float(metrics.seg.mr),
            "map50": float(metrics.seg.map50),
            "map50_95": float(metrics.seg.map),
        },
    }
    (out_dir / "metrics.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print(name, json.dumps(summary, indent=2, ensure_ascii=False))
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return summary


def eval_pixel(model_path, data_yaml, name, split="test", conf=0.25, iou=0.7):
    from ultralytics import YOLO
    data = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
    root = Path(data.get("path", Path(data_yaml).parent))
    if not root.is_absolute():
        root = (Path(data_yaml).parent / root).resolve()
    split_rel = str(data.get(split))
    images_dir = Path(split_rel)
    if not images_dir.is_absolute():
        images_dir = root / images_dir
    labels_dir = Path(split_rel.replace("images", "labels", 1))
    if not labels_dir.is_absolute():
        labels_dir = root / labels_dir

    names = load_names(data_yaml)
    num_classes = len(names)
    model = YOLO(str(model_path))
    matrix = np.zeros((num_classes + 1, num_classes + 1), dtype=np.int64)
    images = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

    for i, image_path in enumerate(images, 1):
        img = cv2.imread(str(image_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        gt = yolo_txt_to_mask(labels_dir / f"{image_path.stem}.txt", h, w, num_classes)
        result = model.predict(str(image_path), imgsz=IMG_SIZE, conf=conf, iou=iou, device=DEVICE, verbose=False)[0]
        pred = np.zeros((h, w), dtype=np.uint8)
        if result.masks is not None and result.boxes is not None:
            classes = result.boxes.cls.cpu().numpy().astype(int)
            masks = result.masks.data.cpu().numpy()
            for cls_id, mask in zip(classes, masks):
                if 0 <= cls_id < num_classes:
                    resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
                    pred[resized > 0.5] = cls_id + 1
        flat = gt.astype(np.int64) * (num_classes + 1) + pred.astype(np.int64)
        matrix += np.bincount(flat.ravel(), minlength=(num_classes + 1) ** 2).reshape((num_classes + 1, num_classes + 1))
        if i % 50 == 0:
            print(f"  {name}: {i}/{len(images)}")

    out_dir = RESULTS / name
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for cid, cname in enumerate(names):
        tid = cid + 1
        tp = int(matrix[tid, tid])
        fn = int(matrix[tid, :].sum() - tp)
        fp = int(matrix[:, tid].sum() - tp)
        tn = int(matrix.sum() - tp - fn - fp)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0
        rows.append({
            "class_id": cid, "class_name": cname,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "accuracy": round(accuracy, 6),
            "precision": round(precision, 6),
            "recall": round(recall, 6),
            "f1": round(f1, 6),
        })

    with (out_dir / "pixel_metrics.csv").open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    labels = ["background"] + names
    with (out_dir / "pixel_confusion_matrix.csv").open("w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["true\\pred"] + labels)
        for i, label in enumerate(labels):
            writer.writerow([label] + matrix[i].tolist())

    print(f"Pixel evaluation saved: {out_dir}")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return rows

## 13. Evaluasi external test

In [ ]:
if RUN_EXTERNAL_EVAL:
    main_best = RUNS / "seg_main" / "weights" / "best.pt"
    main_last = RUNS / "seg_main" / "weights" / "last.pt"
    model_path = main_best if main_best.exists() else main_last
    print("Model evaluasi:", model_path)

    if not model_path.exists():
        print("Model tidak ditemukan. Jalankan training dulu atau set RUN_MAIN=True.")
    else:
        external = {
            "uec_test": EXTERNAL_ROOT / "uec_test" / "data.yaml",
            "foodseg_val": EXTERNAL_ROOT / "foodseg_val" / "data.yaml",
            "fruitseg": EXTERNAL_ROOT / "fruitseg" / "data.yaml",
        }
        for name, yaml_path in external.items():
            if not Path(yaml_path).is_file():
                print("Skip, data.yaml tidak ada:", yaml_path)
                continue
            print("\n" + "=" * 70)
            print("External test:", name)
            print("=" * 70)
            eval_instance(model_path, yaml_path, f"{name}_instance")
            eval_pixel(model_path, yaml_path, f"{name}_pixel")
else:
    print("RUN_EXTERNAL_EVAL=False — dilewati.")

## 14. Evaluasi 10–20 foto piring Indonesia (opsional)

Siapkan Kaggle Dataset `indonesian-plates` dengan salah satu struktur:

```
indonesian-plates/
├── images/          # foto jpg
└── labels/          # YOLO polygon .txt (class x1 y1 ...)
```

atau LabelMe:

```
indonesian-plates/
├── images/          # foto jpg
└── labelme/         # JSON LabelMe
```

Label kelas yang diterima: `Staple Food`, `Lainnya`, `Makanan Pokok`,
`Lauk Pauk`, `Sayuran`, `Buah-buahan` (case-insensitive).

Dengan 10–20 gambar, mAP tidak stabil secara statistik. Gunakan hasil ini
sebagai **exploratory/qualitative domain-shift analysis**, bukan benchmark.

In [ ]:
LABEL_TO_ID = {
    "staple food": 0, "makanan pokok": 0, "karbohidrat": 0,
    "side dishes": 1, "lauk pauk": 1, "lauk-pauk": 1, "protein": 1,
    "vegetables": 2, "sayuran": 2, "sayur": 2,
    "fruits": 3, "buah-buahan": 3, "buah": 3,
    "others": 4, "lainnya": 4, "other": 4,
}


def convert_labelme(json_dir, image_dir, out_root):
    out_img = out_root / "test" / "images"
    out_lbl = out_root / "test" / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    count = 0
    for json_path in sorted(Path(json_dir).glob("*.json")):
        data = json.loads(json_path.read_text(encoding="utf-8"))
        image_name = data.get("imagePath") or f"{json_path.stem}.jpg"
        src_img = Path(image_dir) / image_name
        if not src_img.is_file():
            src_img = json_path.parent / image_name
        img = None
        if src_img.is_file():
            img = Image.open(src_img).convert("RGB")
        elif data.get("imageData"):
            import base64, io
            img = Image.open(io.BytesIO(base64.b64decode(data["imageData"]))).convert("RGB")
        if img is None:
            continue
        w, h = img.size
        lines = []
        for shape in data.get("shapes", []):
            label = str(shape.get("label", "")).strip().lower()
            cls_id = LABEL_TO_ID.get(label)
            if cls_id is None:
                continue
            points = shape.get("points", [])
            if shape.get("shape_type") == "rectangle" and len(points) == 2:
                (x1, y1), (x2, y2) = points
                points = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
            if len(points) < 3:
                continue
            coords = []
            for x, y in points:
                coords.append(f"{max(0.0, min(1.0, x / w)):.6f} {max(0.0, min(1.0, y / h)):.6f}")
            lines.append(f"{cls_id} " + " ".join(coords))
        out_stem = f"idn_{json_path.stem}"
        img.save(out_img / f"{out_stem}.jpg", quality=95)
        (out_lbl / f"{out_stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        count += 1
    print("LabelMe converted:", count)
    return count


if RUN_INDONESIAN_EVAL and INDONESIAN_ROOT.is_dir():
    indo_root = EXTERNAL_ROOT / "indonesian"
    labelme_dir = INDONESIAN_ROOT / "labelme"
    images_dir = INDONESIAN_ROOT / "images"
    labels_dir = INDONESIAN_ROOT / "labels"

    if labelme_dir.is_dir():
        convert_labelme(labelme_dir, images_dir, indo_root)
    elif images_dir.is_dir() and labels_dir.is_dir():
        shutil.copytree(images_dir, indo_root / "test" / "images", dirs_exist_ok=True)
        shutil.copytree(labels_dir, indo_root / "test" / "labels", dirs_exist_ok=True)
    else:
        print("Struktur Indonesian dataset tidak dikenali.")
        indo_root = None

    if indo_root and (indo_root / "test" / "images").is_dir():
        write_data_yaml(indo_root / "data.yaml", indo_root, "test/images", "test/images", "test/images")
        model_path = RUNS / "seg_main" / "weights" / "best.pt"
        if not model_path.exists():
            model_path = RUNS / "seg_main" / "weights" / "last.pt"
        if not model_path.exists():
            print("Model tidak ditemukan, evaluasi Indonesia dilewati.")
        else:
            eval_instance(model_path, indo_root / "data.yaml", "indonesian_exploratory_instance")
            eval_pixel(model_path, indo_root / "data.yaml", "indonesian_exploratory_pixel")
else:
    print("RUN_INDONESIAN_EVAL=False atau folder tidak ada — dilewati.")

## 15. Package hasil untuk diunduh

In [ ]:
zip_path = WORK / "revision_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (RUNS, RESULTS):
        if folder.is_dir():
            for path in folder.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(WORK))
    for extra in (
        DATA_ROOT / "data.yaml",
        DATA_ROOT / "dataset_summary.json",
        DATA_ROOT / "manifest.json",
        DATA_ROOT / "training_config.json",
    ):
        if extra.is_file():
            zf.write(extra, extra.relative_to(WORK))

print("ZIP:", zip_path)
print("Main best:", RUNS / "seg_main" / "weights" / "best.pt")
print("Main last:", RUNS / "seg_main" / "weights" / "last.pt")
print("Results:", RESULTS)
print("\nDownload dari tab Output Kaggle:")
print(" - revision_results.zip")
print(" - runs/seg_main/weights/best.pt")

In [ ]:
"""
backup_all.py — amankan SEMUA hasil training/evaluasi dalam satu paket terverifikasi.

Cara pakai di Kaggle: tempel isi file ini ke cell baru PALING AKHIR notebook
(atau jalankan: exec(open("backup_all.py").read())).

Hasil:
  /kaggle/working/backup/
    results_backup_<tanggal>.zip      <- hasil training, evaluasi, label, config, environment
    dataset_images_<tanggal>.zip      <- (opsional) gambar dataset, jika INCLUDE_IMAGES=True
    CHECKSUMS.sha256                  <- hash semua file (untuk verifikasi setelah download)
    FILE_LIST.csv                     <- daftar file + ukuran
"""
import os, sys, csv, json, shutil, hashlib, subprocess, zipfile, datetime
from pathlib import Path

# ── Pengaturan ───────────────────────────────────────────────────────────────
WORK = Path(os.environ.get("BACKUP_WORK", "/kaggle/working"))
TEMP = Path(os.environ.get("BACKUP_TEMP", "/kaggle/temp"))
INCLUDE_IMAGES = False   # True = ikut arsipkan gambar dataset (besar, beberapa GB)
OUT = WORK / "backup"
STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M")
SKIP_SUFFIX = {".npy", ".cache"}  # cache ultralytics, bisa dibuat ulang

OUT.mkdir(parents=True, exist_ok=True)


def find_dir(name):
    """Cari folder di /kaggle/working dulu, lalu /kaggle/temp (tergantung CONFIG)."""
    for base in (WORK, TEMP):
        if (base / name).is_dir():
            return base / name
    return None


RUNS = find_dir("runs")
RESULTS = find_dir("results")
DATA_ROOT = find_dir("revision_dataset")
EXTERNAL_ROOT = find_dir("external")


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def environment_snapshot():
    env_dir = OUT / "_env"
    env_dir.mkdir(exist_ok=True)
    (env_dir / "python_version.txt").write_text(sys.version)
    for fname, cmd in (
        ("pip_freeze.txt", [sys.executable, "-m", "pip", "freeze"]),
        ("nvidia_smi.txt", ["nvidia-smi"]),
    ):
        try:
            res = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            (env_dir / fname).write_text(res.stdout + res.stderr)
        except Exception as exc:
            (env_dir / fname).write_text(f"gagal: {exc}")
    return env_dir


# ── Kumpulkan file yang diarsipkan (path di zip, path asli) ──────────────────
entries = []


def add_tree(root, arc_prefix, only_suffix=None, only_dirs=None):
    if root is None:
        return
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() in SKIP_SUFFIX:
            continue
        if only_suffix and p.suffix.lower() not in only_suffix:
            continue
        if only_dirs and not any(part in only_dirs for part in p.parts):
            continue
        entries.append((f"{arc_prefix}/{p.relative_to(root).as_posix()}", p))


add_tree(RUNS, "runs")           # best.pt, last.pt, results.csv, args.yaml, plot, dll
add_tree(RESULTS, "results")     # semua evaluasi instance + pixel (json, csv, png, predictions.json)

# Dataset: SEMUA label (txt) + yaml + json (manifest, summary, config) — tanpa gambar
add_tree(DATA_ROOT, "revision_dataset", only_suffix={".txt", ".yaml", ".yml", ".json"})
add_tree(EXTERNAL_ROOT, "external", only_suffix={".txt", ".yaml", ".yml", ".json"})

# Notebook & zip lama kalau ada
for extra in list(WORK.glob("*.ipynb")) + [WORK / "revision_results.zip"]:
    if extra.is_file():
        entries.append((f"extra/{extra.name}", extra))

env_dir = environment_snapshot()
for p in env_dir.iterdir():
    entries.append((f"environment/{p.name}", p))

# ── Tulis zip utama ──────────────────────────────────────────────────────────
main_zip = OUT / f"results_backup_{STAMP}.zip"
with zipfile.ZipFile(main_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for arc, src in entries:
        zf.write(src, arc)
print(f"[1/4] Zip utama: {main_zip.name} ({len(entries)} file, {main_zip.stat().st_size/1e6:.1f} MB)")

# ── Zip gambar (opsional) ────────────────────────────────────────────────────
image_zip = None
if INCLUDE_IMAGES:
    image_zip = OUT / f"dataset_images_{STAMP}.zip"
    n_img = 0
    with zipfile.ZipFile(image_zip, "w", zipfile.ZIP_STORED) as zf:  # jpg sudah terkompresi
        for root, prefix in ((DATA_ROOT, "revision_dataset"), (EXTERNAL_ROOT, "external")):
            if root is None:
                continue
            for p in root.rglob("*"):
                if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    zf.write(p, f"{prefix}/{p.relative_to(root).as_posix()}")
                    n_img += 1
    print(f"[2/4] Zip gambar: {image_zip.name} ({n_img} gambar, {image_zip.stat().st_size/1e9:.2f} GB)")
else:
    print("[2/4] Gambar dataset tidak diarsipkan (INCLUDE_IMAGES=False). "
          "Dataset bisa dibuat ulang dari notebook dengan SEED=42 + manifest.json.")

# ── Verifikasi: zip tidak korup & semua file masuk ───────────────────────────
for z in [main_zip] + ([image_zip] if image_zip else []):
    with zipfile.ZipFile(z) as zf:
        bad = zf.testzip()
        assert bad is None, f"ZIP korup di: {bad}"
with zipfile.ZipFile(main_zip) as zf:
    in_zip = set(zf.namelist())
missing = [arc for arc, _ in entries if arc not in in_zip]
assert not missing, f"File hilang dari zip: {missing[:5]}"
print("[3/4] Verifikasi zip OK (tidak korup, semua file masuk)")

# ── Checksum + daftar file ───────────────────────────────────────────────────
with open(OUT / "CHECKSUMS.sha256", "w") as f:
    for z in [main_zip] + ([image_zip] if image_zip else []):
        f.write(f"{sha256(z)}  {z.name}\n")
with open(OUT / "FILE_LIST.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["path_in_zip", "size_bytes"])
    for arc, src in entries:
        w.writerow([arc, src.stat().st_size])
shutil.rmtree(env_dir, ignore_errors=True)
print("[4/4] CHECKSUMS.sha256 & FILE_LIST.csv ditulis")

# ── Ringkasan: pastikan komponen penting ada ─────────────────────────────────
must_have = {
    "best.pt": "runs/seg_main/weights/best.pt",
    "last.pt": "runs/seg_main/weights/last.pt",
    "results.csv": "runs/seg_main/results.csv",
    "args.yaml": "runs/seg_main/args.yaml",
    "dataset_summary.json": "revision_dataset/dataset_summary.json",
    "manifest.json": "revision_dataset/manifest.json",
    "uec_test instance": "results/uec_test_instance/metrics.json",
    "uec_test pixel": "results/uec_test_pixel/pixel_metrics.csv",
    "foodseg_val instance": "results/foodseg_val_instance/metrics.json",
    "foodseg_val pixel": "results/foodseg_val_pixel/pixel_metrics.csv",
    "fruitseg instance": "results/fruitseg_instance/metrics.json",
    "fruitseg pixel": "results/fruitseg_pixel/pixel_metrics.csv",
}
print("\nCek komponen penting:")
for label, arc in must_have.items():
    print(f"  {'OK     ' if arc in in_zip else 'HILANG '} {label}")

print(f"""
Langkah berikutnya (PENTING, supaya benar-benar aman):
 1. Klik 'Save Version' -> 'Save & Run All (Commit)'. Output hanya tersimpan permanen
    untuk versi yang di-commit.
 2. Di halaman versi itu: tab Output -> 'New Dataset' agar tersimpan sebagai
    Kaggle Dataset pribadi (bisa dipakai ulang sebagai input tanpa training lagi).
 3. Download juga {main_zip.name} + CHECKSUMS.sha256 ke komputer, lalu salin ke
    Google Drive / flashdisk. Verifikasi: sha256sum -c CHECKSUMS.sha256
""")

## 16. Checklist sebelum submit

- [ ] `dataset_summary.json` menunjukkan train/val/test dan distribusi kelas.
- [ ] `Fruits` ada di train (dari FoodSeg). Jika 0, pindahkan sebagian FruitSeg30 ke train.
- [ ] `best.pt`, `last.pt`, `results.csv`, `args.yaml` diunduh.
- [ ] Instance-level dan pixel-level metrics dipisahkan di paper.
- [ ] External test (UEC, FoodSeg, FruitSeg) dilaporkan per sumber.
- [ ] 10–20 foto Indonesia dilaporkan sebagai exploratory, bukan benchmark.
- [ ] OCR eval + ablation dijalankan terpisah (`experiments/scripts/eval_ocr.py`,
      `ablate_ocr.py`).
- [ ] Semua angka di abstract = results = tables.